# Log–log plot of the Holevo-information gap

Numerics for the section **"Optimization of the Holevo information"** of
*Quantum Programmable Reflections* (Schoute, Grinko, Subaşı, Volkoff).

This notebook loads the pre-computed Holevo information of the symmetric-subspace probe state
(produced by `symm_subspace_holevo_information_d_gtreq_3.ipynb`) and plots the gap to the upper
bound, $1 - r$ with $r = S(\widetilde{\mathcal T(\rho)}) / (2\log_2 D(n,d))$, on a log–log scale.
The gap behaves as $c(d)\,n^{-\alpha}$ with $\alpha \approx 0.3$, roughly independent of $d$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import comb

## Load the pre-computed Holevo information

In [ ]:
DATA_FILE = "holevo_information_data_d_3_20_n_1_41.pkl"
dims   = np.arange(3, 21)    # d = 3, …, 20
copies = np.arange(1, 41)    # n = 1, …, 40

holevo = np.asarray(pd.read_pickle(DATA_FILE))   # shape (len(dims), len(copies))
assert holevo.shape == (len(dims), len(copies)), holevo.shape

## Ratio to the upper bound

In [ ]:
# Upper bound 2 log2 binomial(n + d - 1, d - 1), broadcast over (d, n), and the ratio r.
upper_bound = 2 * np.log2(comb(copies[None, :] + dims[:, None] - 1, dims[:, None] - 1))
ratio = holevo / upper_bound

## Log–log plot of the gap $1 - r$

$d = 3$ is omitted: the bound is already saturated at $n = 1$, so $1 - r = 0$ there and its
logarithm diverges.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for i, d in enumerate(dims):
    if d == 3:
        continue
    gap = 1.0 - ratio[i]
    ax.plot(np.log(copies), np.log(gap), marker=".", label=f"d = {d}")
ax.set_xlabel(r"$\log n$")
ax.set_ylabel(r"$\log\,(1 - r)$")
ax.set_title(r"gap to the upper bound:  $1 - r \sim c(d)\, n^{-\alpha}$")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig("ratfig.pdf")
fig

In [ ]:
# Estimate the exponent α from a linear fit of log(1 - r) vs log n over the larger n.
fit_from = 10   # index into `copies`
for i, d in enumerate(dims):
    if d == 3:
        continue
    slope, _ = np.polyfit(np.log(copies[fit_from:]), np.log(1.0 - ratio[i, fit_from:]), 1)
    print(f"d = {d:2d}:   alpha ~ {-slope:.3f}")